# 01 — Data Profiling

## GulfMart Retail Inventory Analytics

This notebook profiles the generated dataset in `data/raw/` before any
cleaning happens. It answers: **what does this data actually look like,
and what's messy about it?**

Profiling logic lives in `src/data_profiling.py` (missingness, duplicates,
text consistency). Correctness checks live in `src/data_validation.py`
(structural, financial, business-rule, inventory-policy validation). This
notebook imports both rather than re-implementing them, so the checks
that run here are exactly the same checks that run during data generation.

**Pipeline order in this notebook is deliberate:**

```text
Load small tables -> profile
Load fact_inventory (full precision as read from disk)
Validate fact_inventory (before any dtype changes)
Downcast fact_inventory dtypes (for memory efficiency)
Profile fact_inventory (on the downcast data)
Descriptive statistics
Key business-shape questions
Findings summary -> feeds directly into 02_data_cleaning.ipynb
```

Validation runs **before** downcasting on purpose: correctness should
always be checked against the most trustworthy version of the data
available in this notebook. Downcasting is applied afterward, for the
profiling/analysis steps that follow, where a small amount of
floating-point rounding is an acceptable trade-off for materially lower
memory usage.


## 1. Setup

Make `src/` importable and resolve the `data/raw/` path relative to the
project root rather than the notebook's own folder, since Jupyter's
working directory is wherever the `.ipynb` file lives (`notebooks/`).

In [11]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent  # notebooks/ -> project root
SRC_DIR = PROJECT_ROOT / "src"
DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import pandas as pd

from data_profiling import profile_all_tables
from data_validation import validate_generated_dataset
from utils import downcast_dtypes

print(f"Project root: {PROJECT_ROOT}")
print(f"Reading data from: {DATA_RAW_DIR}")


Project root: d:\GitHub\retail-inventory-analytics
Reading data from: d:\GitHub\retail-inventory-analytics\data\raw


## 2. Load & Profile the Small Tables

Dimension tables and `fact_sales` are small enough to load and profile
directly. `profile_all_tables()` reports, per table: shape, memory usage,
missing values, duplicate rows (full-row and on the table's business key),
and text-formatting consistency across every object/string column.

In [12]:
dim_product = pd.read_csv(DATA_RAW_DIR / "dim_product.csv")
dim_customer = pd.read_csv(DATA_RAW_DIR / "dim_customer.csv")
dim_supplier = pd.read_csv(DATA_RAW_DIR / "dim_supplier.csv")
dim_store = pd.read_csv(DATA_RAW_DIR / "dim_store.csv")
fact_sales = pd.read_csv(DATA_RAW_DIR / "fact_sales.csv")

small_table_profiles = profile_all_tables({
    "dim_product": dim_product,
    "dim_customer": dim_customer,
    "dim_supplier": dim_supplier,
    "dim_store": dim_store,
    "fact_sales": fact_sales,
})



############################################################
# PROFILING: dim_product
############################################################

Shape: 500 rows x 14 columns
Memory usage: 0.30 MB

dim_product — Missingness Profile
No missing values detected.

dim_product — Duplicate Profile
Full-row duplicates: 0
Duplicate on key product_id: 0

dim_product — Text Consistency Profile
No text-consistency issues detected.

############################################################
# PROFILING: dim_customer
############################################################

Shape: 5,000 rows x 11 columns
Memory usage: 2.03 MB

dim_customer — Missingness Profile
column  missing_count  missing_pct
  city              4         0.08

dim_customer — Duplicate Profile
Full-row duplicates: 0
Duplicate on key customer_id: 0

dim_customer — Text Consistency Profile
No text-consistency issues detected.

############################################################
# PROFILING: dim_supplier
#########

## 3. Load `fact_inventory`

The largest table (~10.96M rows). Loaded at whatever precision
`pd.read_csv()` infers from the saved CSV, with load time reported since
this is the slowest single operation in the notebook.

In [13]:
import time

start = time.time()

fact_inventory = pd.read_csv(DATA_RAW_DIR / "fact_inventory.csv.gz")

print(f"Loaded in {time.time() - start:.1f}s")
print(f"Shape: {fact_inventory.shape[0]:,} rows x {fact_inventory.shape[1]} columns")
print(f"Memory usage: {fact_inventory.memory_usage(deep=True).sum() / 1024**2:.2f} MB")


Loaded in 62.8s
Shape: 10,960,000 rows x 46 columns
Memory usage: 7537.03 MB


## 4. Validation Checkpoint

Runs the complete `validate_generated_dataset()` suite (17 checks:
dimension integrity, fact_sales structure/financials, foreign keys,
inventory structure, duplicates, inventory policy formulas, continuity,
negative-inventory, reconciliation, lost sales, stockouts, replenishment,
coverage, and status) directly in the notebook, so anyone reading this on
GitHub sees live proof the data is structurally sound as part of the
profiling narrative, rather than trusting an invisible console run from
data generation.

Two REVIEW items are expected and not a problem: `inventory_reconciliation`
and `lost_sales` check for columns (`demand_units`, `fulfilled_sales_units`,
`lost_sales_units`) that belong to an earlier row-by-row simulation design
this project no longer uses. The current model tracks `sales_units`
directly and reconciles inventory through the cumulative closing-stock
calculation instead, which the `inventory_continuity` and
`negative_inventory` checks already confirm holds with zero failures.

In [14]:
validation_results = validate_generated_dataset(
    fact_sales,
    fact_inventory,
    dim_product,
    dim_store,
    dim_customer,
    dim_supplier,
)



GulfMart Retail Inventory Dataset Validation

Dataset Summary
Products:       500
Stores:         20
Customers:      5,000
Suppliers:      30
Sales rows:     125,125
Inventory rows: 10,960,000

Product Dimension Validation
PASS: Product Dimension key completeness - NULL product_id values: 0
PASS: Product Dimension key uniqueness - Duplicate product_id values: 0

Store Dimension Validation
PASS: Store Dimension key completeness - NULL store_id values: 0
PASS: Store Dimension key uniqueness - Duplicate store_id values: 0

Customer Dimension Validation
PASS: Customer Dimension key completeness - NULL customer_id values: 0
PASS: Customer Dimension key uniqueness - Duplicate customer_id values: 0

Supplier Dimension Validation
PASS: Supplier Dimension key completeness - NULL supplier_id values: 0
PASS: Supplier Dimension key uniqueness - Duplicate supplier_id values: 0

Fact Sales Validation
PASS: Required columns - Missing columns: []
FAIL: Transaction ID uniqueness - Duplicate transactio

## 5. Downcast `fact_inventory` Dtypes

`downcast_dtypes()` (in `src/utils.py`) shrinks each numeric column to the
smallest dtype that safely holds every value in it (e.g. `float64` ->
`float32`), using `pd.to_numeric(..., downcast=...)` so it is driven by
the actual data range rather than a blind cast.

Note this is the **second** call to `downcast_dtypes()` for this data —
the first happens inside `save_datasets()` during generation, before
writing to CSV. That first call only reduces peak RAM *while saving*; it
does not shrink the CSV file itself, since CSV stores every value as
plain text regardless of the underlying dtype. `pd.read_csv()` always
re-infers `float64`/`int64` by default, discarding whatever dtype the
data had before — so this second call is what actually delivers the
memory benefit for everything that follows in this notebook.

In [15]:
fact_inventory = downcast_dtypes(fact_inventory)


Memory usage: 7537.03 MB -> 5781.05 MB (23.3% reduction)


## 6. Profile `fact_inventory`

Same profiling suite as Section 2, applied to the downcast inventory
table. Its uniqueness key is the combination of `date`, `store_id`, and
`product_id` (there is no single-column ID for this table).

In [16]:
fact_inventory_profile = profile_all_tables({"fact_inventory": fact_inventory})



############################################################
# PROFILING: fact_inventory
############################################################

Shape: 10,960,000 rows x 46 columns
Memory usage: 5781.05 MB

fact_inventory — Missingness Profile
                 column  missing_count  missing_pct
inventory_coverage_days        2383800        21.75

fact_inventory — Duplicate Profile
Full-row duplicates: 0
Duplicate on key ['date', 'store_id', 'product_id']: 0

fact_inventory — Text Consistency Profile
No text-consistency issues detected.

PROFILING COMPLETE
fact_inventory  10,960,000 rows   5781.05 MB


## 7. Descriptive Statistics

Numeric distributions for the key financial/quantity fields, and value
counts for the categorical fields that drive the demand model
(demand class, category, store type, customer segment, inventory
status).

In [17]:
print("fact_sales -- key numeric fields")
print(
    fact_sales[
        [
            "quantity",
            "unit_price",
            "gross_sales",
            "discount_pct",
            "net_sales",
            "cogs",
            "gross_profit",
        ]
    ].describe()
)


fact_sales -- key numeric fields
            quantity     unit_price    gross_sales   discount_pct  \
count  125125.000000  125125.000000  125125.000000  125125.000000   
mean        4.422841     349.335758    1536.519584       0.020780   
std         2.907318     225.113587    1570.995060       0.061693   
min         1.000000       6.980000       6.980000       0.000000   
25%         2.000000     140.060000     430.800000       0.000000   
50%         4.000000     330.110000     969.300000       0.000000   
75%         7.000000     536.940000    2092.560000       0.000000   
max        10.000000     879.930000    7924.800000       0.300000   

           net_sales           cogs   gross_profit  
count  125125.000000  125125.000000  125125.000000  
mean     1500.618668    1056.556629     444.062039  
std      1534.910699    1069.837420     529.390456  
min         5.800000       5.410000    -720.800000  
25%       424.160000     300.580000     100.550000  
50%       942.450000     66

In [18]:
print("dim_product -- demand_class distribution")
print(dim_product["demand_class"].value_counts())
print()

print("dim_product -- category distribution")
print(dim_product["category"].value_counts())
print()

print("dim_store -- store_type distribution")
print(dim_store["store_type"].value_counts())
print()

print("dim_customer -- customer_segment distribution")
print(dim_customer["customer_segment"].value_counts())
print()

print("fact_inventory -- inventory_status distribution")
print(fact_inventory["inventory_status"].value_counts())


dim_product -- demand_class distribution
demand_class
Medium-moving    271
Slow-moving      137
Fast-moving       92
Name: count, dtype: int64

dim_product -- category distribution
category
Grocery          114
Electronics       59
Household         58
Beauty            57
Beverages         56
Fashion           53
Home & Living     53
Personal Care     50
Name: count, dtype: int64

dim_store -- store_type distribution
store_type
Supermarket    9
Express        5
Hypermarket    5
E-commerce     1
Name: count, dtype: int64

dim_customer -- customer_segment distribution
customer_segment
Regular    2520
Value      1248
Premium     741
New         491
Name: count, dtype: int64

fact_inventory -- inventory_status distribution
inventory_status
Healthy      10926211
Low Stock       33789
Name: count, dtype: int64


## 8. Key Business-Shape Questions

A handful of `groupby`s to surface what's interesting about the data's
shape before analysis begins -- not conclusions yet, just observations to
carry forward.

In [19]:
print("Sales date coverage")
print(f"Min: {fact_sales['transaction_date'].min()}  Max: {fact_sales['transaction_date'].max()}")
print()

print("Transactions per store (top 5)")
txn_per_store = (
    fact_sales.groupby("store_id")["transaction_id"].count().sort_values(ascending=False)
)
print(txn_per_store.head())
print()
print("Transactions per store (bottom 5)")
print(txn_per_store.tail())


Sales date coverage
Min: 2023-01-01  Max: 2025-12-31

Transactions per store (top 5)
store_id
STORE014    10214
STORE015    10123
STORE016     8694
STORE009     8013
STORE019     7861
Name: transaction_id, dtype: int64

Transactions per store (bottom 5)
store_id
STORE012    4152
STORE013    3551
STORE001    2846
STORE008    2686
STORE010    2683
Name: transaction_id, dtype: int64


In [20]:
print("Transactions per product (top 5)")
txn_per_product = (
    fact_sales.groupby("product_id")["transaction_id"].count().sort_values(ascending=False)
)
print(txn_per_product.head())
print()

all_products = set(dim_product["product_id"].unique())
products_with_sales = set(fact_sales["product_id"].unique())
zero_sales_products = all_products - products_with_sales

print(f"Products with zero recorded sales: {len(zero_sales_products):,} of {len(all_products):,}")


Transactions per product (top 5)
product_id
PROD0075    1798
PROD0426    1768
PROD0320    1673
PROD0062    1626
PROD0277    1572
Name: transaction_id, dtype: int64

Products with zero recorded sales: 4 of 500


In [21]:
all_combos = fact_inventory[["store_id", "product_id"]].drop_duplicates()

combos_with_sales = (
    fact_inventory.loc[fact_inventory["sales_units"] > 0, ["store_id", "product_id"]]
    .drop_duplicates()
)

print(
    f"Product-store combinations with at least one recorded sale: "
    f"{len(combos_with_sales):,} of {len(all_combos):,} "
    f"({len(combos_with_sales) / len(all_combos):.1%})"
)


Product-store combinations with at least one recorded sale: 7,825 of 10,000 (78.2%)


## 9. Findings Summary

Plain-language summary of what this profiling pass found. This becomes
the task list for `02_data_cleaning.ipynb` -- update the counts below
after each run, since the data-quality injection in `inject_data_quality_issues()`
is probability-driven and can vary slightly run to run.

**Data-quality issues to clean (planted intentionally, in `02_data_cleaning.ipynb`):**

- `dim_customer.city` -- missing values (last observed: 4 rows)
- `dim_supplier.supplier_name` -- missing values (last observed: 1 row)
- `fact_sales` -- duplicate transactions, full-row and on `transaction_id`
  (last observed: 125 rows)
- `dim_store.city` -- inconsistent text formatting (case / whitespace);
  probability-driven on a 20-row table, so this may show 0 on some runs

**Legitimate business findings (not data-quality issues -- carry forward
into analysis, do not "fix"):**

- A large share of product-store combinations never recorded a sale in
  the transaction data (roughly a fifth, based on prior profiling runs).
  This is a genuine sparsity characteristic of the synthetic demand
  model, not missing data to impute.
- The majority of inventory rows show very high coverage-days
  (>365 days for the large majority of rows with a defined coverage
  value). This reflects well-calibrated safety stock producing very few
  real stockouts -- worth flagging explicitly for discussion in the
  eventual Stockout Analysis notebook rather than treated as an error.
- `inventory_reconciliation` and `lost_sales` validation checks report
  REVIEW, not PASS/FAIL -- expected, since this project's current
  inventory model does not track `demand_units` / `fulfilled_sales_units`
  / `lost_sales_units` as separate fields (an earlier row-by-row
  simulation design did; the current vectorized model reconciles through
  cumulative closing stock instead, which the continuity check already
  confirms holds exactly).

**Next step:** `02_data_cleaning.ipynb` resolves the data-quality issues
listed above and re-validates afterward.